# 22 Holdout LMS Pendikom


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
from io import BytesIO
import os, sys, json, shutil, zipfile, subprocess, hashlib, itertools, importlib.util, csv, warnings
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import CategoricalNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

PROJECT_ROOT = _ROOT_REVISED
DATA_DIR = PROJECT_ROOT / 'DATA'
PYTHON_DIR = PROJECT_ROOT / 'PYTHON'
LIB_DIR = PROJECT_ROOT / 'LIB'
OUTPUT_DIR = stage_dir('22_Holdout_LMS_Pendikom')
DATA_DIR.mkdir(parents=True, exist_ok=True)
PYTHON_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))
sys.modules.pop('care_km', None)
import care_km as ck

PENDIKOM_CATEGORICAL_INPUT = pendikom_data('categorical')
GROUNDTRUTH_SPLIT = _ROOT_REVISED / 'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_split_audit.csv'
SPLIT_SEED = 161600
TEST_SIZE = 0.30
K = 16
MAX_ITER = 100
CARE_CONFIG = dict(candidate_rule='separation', candidate_budget=16, min_gain=0.0)
DERIVED_REFINEMENT_BUDGET = ck.derive_refinement_budget(CARE_CONFIG['candidate_budget'])
assert DERIVED_REFINEMENT_BUDGET == 4
STAGE14_DERIVED_VERIFY_VERSION = 'source-bound-lib-carekm-v6'

MAIN_FEATURES = [
    'content_visit','content_stay','selfass_visit_proxy','selfass_stay_proxy',
    'exercise_visit_proxy','exercise_stay_proxy','forum_visit','forum_stay','forum_post',
    'quiz_stay_results_proxy','navigation_overview_visit','navigation_overview_stay','return_to_overview_rate'
]

DIM_NAMES = ['Active / Reflective','Sensing / Intuitive','Visual / Verbal','Sequential / Global']
DIM_KEYS = ['active_reflective','sensing_intuitive','visual_verbal','sequential_global']

print('PROJECT_ROOT   :', PROJECT_ROOT)
print('PENDIKOM_CATEGORICAL_INPUT  :', PENDIKOM_CATEGORICAL_INPUT)
print('GROUNDTRUTH_SPLIT:', GROUNDTRUTH_SPLIT)
print('OUTPUT_DIR     :', OUTPUT_DIR)
print('STAGE14 DERIVED VERIFY:', STAGE14_DERIVED_VERIFY_VERSION)
print('CARE implementation:', ck.CARE_IMPLEMENTATION_VERSION)
print('Derived refinement budget:', DERIVED_REFINEMENT_BUDGET)


# Stage-14 v6 source binding: use the same canonical CARE-KM implementation
# as the main pipeline. This prevents notebook-local algorithm drift.
assert ck.CARE_IMPLEMENTATION_VERSION == 'encoded-incremental-derived-budget-v3'
assert ck.derive_refinement_budget(CARE_CONFIG['candidate_budget']) == DERIVED_REFINEMENT_BUDGET
print('Stage-14 CARE-KM source binding: PASS')


POLES = {
    'Active/Reflective': ('Active','Reflective'),
    'Sensing/Intuitive': ('Sensing','Intuitive'),
    'Visual/Verbal': ('Visual','Verbal'),
    'Sequential/Global': ('Sequential','Global'),
}
ABBR = {
    'Active':'Act.','Reflective':'Ref.','Sensing':'Sens.','Intuitive':'Int.',
    'Visual':'Vis.','Verbal':'Ver.','Sequential':'Seq.','Global':'Gl.'
}
DIMENSION_ORDER = list(POLES.keys())
LS_LOOKUP = {tuple(combo): f'LS{i:02d}' for i, combo in enumerate(itertools.product(*[POLES[d] for d in DIMENSION_ORDER]), start=1)}
LS_TO_POLES = {v:k for k,v in LS_LOOKUP.items()}


def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()


def one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=True)


def categorical_nb():
    try:
        return CategoricalNB(alpha=1.0, min_categories=4)
    except TypeError:
        return CategoricalNB(alpha=1.0)

def build_main_classifiers():
    # Prospective main-LMS configuration.
    return {
        'NB': categorical_nb(),
        'DT': Pipeline([('onehot', one_hot_encoder()), ('model', DecisionTreeClassifier(random_state=SPLIT_SEED))]),
        'SVM': Pipeline([('onehot', one_hot_encoder()), ('model', SVC(kernel='rbf', C=1.0, gamma='scale'))]),
    }

def build_external_classifiers():
    # Frozen configuration used by the locked ground-truth Table 2.
    return {
        'NB': categorical_nb(),
        'DT': DecisionTreeClassifier(random_state=SPLIT_SEED, class_weight='balanced'),
        'SVM': Pipeline([('onehot', one_hot_encoder()), ('svc', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced'))]),
    }


def fit_care(X):
    X=np.asarray(X,dtype=object)
    init=ck.care_init(X,K,**CARE_CONFIG)
    result=ck.kmodes_fit(X,init,max_iter=MAX_ITER)
    return init,result


def signed_vote(value, rule, features):
    value=int(value)
    first,second=POLES[str(rule.dimension)]
    if value==2:
        return 0
    if hasattr(rule,'high_occurrence_indicates'):
        pole = str(rule.high_occurrence_indicates) if value==3 else str(rule.low_occurrence_indicates)
    else:
        pole = str(rule.high_supports) if value==3 else str(rule.low_supports)
    if pole==first: return 1
    if pole==second: return -1
    raise ValueError(f'Unknown semantic pole: {pole}')


def semantic_mapping(X, labels, modes, features, semantic_rules):
    X=np.asarray(X,dtype=object); labels=np.asarray(labels,dtype=int); modes=np.asarray(modes,dtype=object)
    fpos={f:i for i,f in enumerate(features)}
    rows=[]
    for cid in range(K):
        selected=[]; short=[]
        for dim in DIMENSION_ORDER:
            rules=semantic_rules[semantic_rules['dimension'].astype(str).eq(dim)]
            mv=np.asarray([signed_vote(modes[cid,fpos[str(r.feature)]], r, features) for r in rules.itertuples(index=False)],dtype=int)
            ms=int(np.sum(mv>0)-np.sum(mv<0))
            member_X=X[labels==cid]
            nv=[]
            for r in rules.itertuples(index=False):
                j=fpos[str(r.feature)]
                nv.extend(signed_vote(v,r,features) for v in member_X[:,j])
            nv=np.asarray(nv,dtype=int)
            ns=int(np.sum(nv>0)-np.sum(nv<0)) if len(nv) else 0
            first,second=POLES[dim]
            pole = first if ms>0 else second if ms<0 else first if ns>0 else second if ns<0 else None
            selected.append(pole)
            short.append('Balanced' if pole is None else ABBR[pole])
        unresolved=any(x is None for x in selected)
        rows.append({
            'cluster_id':cid,'cluster_name':f'C{cid+1:02d}',
            'cluster_size':int(np.sum(labels==cid)),
            'active_reflective':short[0], 'sensing_intuitive':short[1],
            'visual_verbal':short[2], 'sequential_global':short[3],
            'ls_class':'AMBIGUOUS' if unresolved else LS_LOOKUP[tuple(selected)],
            'mapping_status':'BALANCED' if unresolved else 'RESOLVED',
        })
    return pd.DataFrame(rows)


def main_semantic_rules():
    rows=[]
    def add(f,d,lo,hi): rows.append({'feature':f,'dimension':d,'low_supports':lo,'high_supports':hi})
    add('content_visit','Active/Reflective','Active','Reflective')
    add('content_stay','Active/Reflective','Active','Reflective')
    add('selfass_visit_proxy','Active/Reflective','Reflective','Active')
    add('selfass_stay_proxy','Active/Reflective','Active','Reflective')
    add('exercise_visit_proxy','Active/Reflective','Reflective','Active')
    add('exercise_stay_proxy','Active/Reflective','Reflective','Active')
    add('forum_visit','Active/Reflective','Active','Reflective')
    add('forum_post','Active/Reflective','Reflective','Active')
    add('quiz_stay_results_proxy','Active/Reflective','Active','Reflective')
    add('content_visit','Sensing/Intuitive','Sensing','Intuitive')
    add('content_stay','Sensing/Intuitive','Sensing','Intuitive')
    add('selfass_visit_proxy','Sensing/Intuitive','Intuitive','Sensing')
    add('selfass_stay_proxy','Sensing/Intuitive','Intuitive','Sensing')
    add('exercise_visit_proxy','Sensing/Intuitive','Intuitive','Sensing')
    add('quiz_stay_results_proxy','Sensing/Intuitive','Intuitive','Sensing')
    add('content_visit','Visual/Verbal','Visual','Verbal')
    add('forum_visit','Visual/Verbal','Visual','Verbal')
    add('forum_stay','Visual/Verbal','Visual','Verbal')
    add('forum_post','Visual/Verbal','Visual','Verbal')
    add('navigation_overview_visit','Sequential/Global','Sequential','Global')
    add('navigation_overview_stay','Sequential/Global','Sequential','Global')
    add('return_to_overview_rate','Sequential/Global','Sequential','Global')
    return pd.DataFrame(rows)


def mapping_to_targets(mapping, cluster_ids):
    mp=mapping.set_index('cluster_id')
    target=pd.DataFrame(index=np.arange(len(cluster_ids)))
    rename={
        'active_reflective':'Active / Reflective',
        'sensing_intuitive':'Sensing / Intuitive',
        'visual_verbal':'Visual / Verbal',
        'sequential_global':'Sequential / Global',
    }
    for col,new in rename.items():
        target[new]=[mp.loc[int(c),col] for c in cluster_ids]
    return target


def pole_abbr_to_full(x):
    m={'Act.':'Active','Ref.':'Reflective','Sens.':'Sensing','Int.':'Intuitive','Vis.':'Visual','Ver.':'Verbal','Seq.':'Sequential','Gl.':'Global'}
    return m.get(str(x), 'Balanced')


def ils_code_to_full(code):
    s=str(code)
    for p,full in [('Act.','Active'),('Ref.','Reflective'),('Sens.','Sensing'),('Int.','Intuitive'),('Vis.','Visual'),('Ver.','Verbal'),('Seq.','Sequential'),('Gl.','Global')]:
        if s.startswith(p): return full
    raise ValueError(f'Unexpected ILS code: {code}')


def evaluate_per_dimension(y_true, y_pred):
    return float(np.mean(np.asarray(y_true,dtype=str)==np.asarray(y_pred,dtype=str)))


def save_confusion(y_true, y_pred, path):
    labs=sorted(set(map(str,y_true))|set(map(str,y_pred)))
    pd.DataFrame(confusion_matrix(y_true,y_pred,labels=labs),index=labs,columns=labs).to_csv(path)

print('Helpers ready.')


# Resolve the numeric learner-level matrix used to build the 6,658-row main LMS dataset.
from pipeline_data import load_lms_holdout
globals().update(load_lms_holdout())

Xtr=main_train[MAIN_FEATURES].astype(str).to_numpy(dtype=object)
Xte=main_test[MAIN_FEATURES].astype(str).to_numpy(dtype=object)
main_init,main_fit=fit_care(Xtr)
main_modes=main_fit['modes']; main_train_cluster=main_fit['labels']
main_test_cluster,main_test_dist=ck.assign_nearest(Xte,main_modes)
main_rules=main_semantic_rules()
main_map=semantic_mapping(Xtr,main_train_cluster,main_modes,MAIN_FEATURES,main_rules)
main_map.to_csv(artifact_path('70_30_main_lms_frozen_cluster_to_fslsm_mapping.csv','22_Holdout_LMS_Pendikom'),index=False)
pd.DataFrame(main_modes,columns=MAIN_FEATURES).assign(cluster_id=np.arange(K)).to_csv(artifact_path('70_30_main_lms_frozen_modes.csv','22_Holdout_LMS_Pendikom'),index=False)

main_y_train=mapping_to_targets(main_map,main_train_cluster).applymap(pole_abbr_to_full)
main_y_test=mapping_to_targets(main_map,main_test_cluster).applymap(pole_abbr_to_full)

rows=[]; pred_out=main_test[['user']].copy(); pred_out['reference_cluster_id']=main_test_cluster; pred_out['distance_to_frozen_mode']=main_test_dist
for model_name,model in build_main_classifiers().items():
    accs=[]
    for dim in DIM_NAMES:
        ytr=main_y_train[dim].astype(str); yte=main_y_test[dim].astype(str)
        if model_name=='NB':
            mdl=build_main_classifiers()['NB']; mdl.fit(main_train[MAIN_FEATURES].astype(int).to_numpy(),ytr); pred=mdl.predict(main_test[MAIN_FEATURES].astype(int).to_numpy())
        else:
            mdl=build_main_classifiers()[model_name]; mdl.fit(main_train[MAIN_FEATURES].astype(int),ytr); pred=mdl.predict(main_test[MAIN_FEATURES].astype(int))
        a=accuracy_score(yte,pred); accs.append(a); pred_out[f'{model_name}_{dim}']=pred
    rows.append({'Classifier':model_name, **{DIM_NAMES[i]:accs[i] for i in range(4)}, 'Mean Dimensional Accuracy':float(np.mean(accs))})

main_table=pd.DataFrame(rows).set_index('Classifier').loc[['NB','DT','SVM']].reset_index()
main_table.to_csv(artifact_path('70_30_main_lms_prospective_dimensional_accuracy_numeric.csv','22_Holdout_LMS_Pendikom'),index=False)
main_paper=main_table.copy()
for c in DIM_NAMES+['Mean Dimensional Accuracy']: main_paper[c]=main_paper[c].map(lambda x:f'{100*x:.2f}%')
main_paper.to_csv(artifact_path('70_30_main_lms_prospective_dimensional_accuracy.csv','22_Holdout_LMS_Pendikom'),index=False)
pred_out.to_csv(artifact_path('70_30_main_lms_prospective_predictions.csv','22_Holdout_LMS_Pendikom'),index=False)
print('Main LMS — prospective dimensional accuracy')
display(main_paper)



# ============================================================
# Dataset 2 — PENDIKOM compiled Moodle + ILS questionnaire
# ============================================================

from pipeline_data import load_pendikom_holdout
globals().update(load_pendikom_holdout())

def pendikom_semantic_rules():
    rows = []
    def add(feature, dimension, low, high):
        rows.append({
            'feature': feature,
            'dimension': dimension,
            'low_supports': low,
            'high_supports': high,
        })

    for f in ['selfass_visit','selfass_twice_wrong','exercise_visit','exercise_stay']:
        add(f, 'Active/Reflective', 'Reflective', 'Active')
    for f in ['content_visit','content_stay','quiz_stay_results']:
        add(f, 'Active/Reflective', 'Active', 'Reflective')

    for f in ['selfass_visit','exercise_visit','ques_detail','ques_facts','quiz_revisions','quiz_stay_results']:
        add(f, 'Sensing/Intuitive', 'Intuitive', 'Sensing')
    for f in ['content_visit','content_stay','ques_concepts','ques_develop']:
        add(f, 'Sensing/Intuitive', 'Sensing', 'Intuitive')

    add('ques_graphics', 'Visual/Verbal', 'Verbal', 'Visual')
    for f in ['content_visit','ques_text']:
        add(f, 'Visual/Verbal', 'Visual', 'Verbal')

    add('ques_detail', 'Sequential/Global', 'Global', 'Sequential')
    for f in ['ques_overview','ques_interpret','ques_develop',
              'navigation_skip','navigation_overview_visit','navigation_overview_stay']:
        add(f, 'Sequential/Global', 'Sequential', 'Global')

    return pd.DataFrame(rows)


semantic_rules = pendikom_semantic_rules()
semantic_rules = semantic_rules[
    semantic_rules['feature'].isin(features)
].copy()
semantic_rules.to_csv(
    artifact_path('70_30_groundtruth_semantic_rules.csv','22_Holdout_LMS_Pendikom'), index=False
)

Xtr_gt = gt_train[features].astype(str).to_numpy(dtype=object)
Xte_gt = gt_test[features].astype(str).to_numpy(dtype=object)

ext_init, ext_fit = fit_care(Xtr_gt)
ext_modes = np.asarray(ext_fit['modes'], dtype=object)
ext_train_cluster = np.asarray(ext_fit['labels'], dtype=int)
ext_test_cluster, ext_test_dist = ck.assign_nearest(Xte_gt, ext_modes)
ext_test_cluster = np.asarray(ext_test_cluster, dtype=int)

ext_map = semantic_mapping(
    Xtr_gt, ext_train_cluster, ext_modes, features, semantic_rules
)
ext_map.to_csv(
    artifact_path('70_30_groundtruth_frozen_cluster_to_ls_mapping.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)
pd.DataFrame(ext_modes, columns=features).assign(
    cluster_id=np.arange(K)
).to_csv(
    artifact_path('70_30_groundtruth_frozen_modes.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)

mp = ext_map.set_index('cluster_id')

test_eval = ils_test[
    ['userid','courseid'] + ILS_CODE_COLS + ['ils_ls_class']
].copy().reset_index(drop=True)
test_eval['reference_cluster_id'] = ext_test_cluster
test_eval['distance_to_frozen_mode'] = ext_test_dist
for c in DIM_KEYS + ['ls_class','mapping_status']:
    test_eval[f'carekm_{c}'] = [
        mp.loc[int(k), c] for k in ext_test_cluster
    ]
test_eval.to_csv(
    artifact_path('70_30_groundtruth_test_carekm_vs_ils_full.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)

BASELINE_METHODS = ['CARE-KM','Bai','Cao','K-Modes++','HR-2023','Huang','Random']
BASELINE_STOCHASTIC = {'K-Modes++','HR-2023','Huang','Random'}
BASELINE_SEED = 4161600

baseline_raw_rows = []
baseline_summary_rows = []

for method in BASELINE_METHODS:
    reps = 15 if method in BASELINE_STOCHASTIC else 1
    method_rows = []

    for rep in range(reps):
        seed = BASELINE_SEED + rep
        row, _, _ = ck.run_method(
            method,
            Xtr_gt,
            None,
            K,
            seed=seed,
            care_config=CARE_CONFIG,
            si_sample_size=None,
            si_seed=seed,
            max_iter=MAX_ITER,
        )
        row = dict(row)
        row['rep'] = rep
        method_rows.append(row)
        baseline_raw_rows.append(row)

    method_df = pd.DataFrame(method_rows)
    baseline_summary_rows.append({
        'Method': method,
        'Runs': reps,
        'Initial cost': float(method_df['initial_cost'].mean()),
        'Final cost': float(method_df['final_cost'].mean()),
        'DBI': float(method_df['dbi'].mean()),
        'SI': float(method_df['si'].mean()),
        'Iterations': float(method_df['iterations'].mean()),
        'DBI SD': float(method_df['dbi'].std(ddof=1)) if reps > 1 else np.nan,
        'SI SD': float(method_df['si'].std(ddof=1)) if reps > 1 else np.nan,
    })

baseline_raw = pd.DataFrame(baseline_raw_rows)
baseline_summary = pd.DataFrame(baseline_summary_rows)

baseline_raw.to_csv(
    artifact_path('70_30_groundtruth_clustering_baseline_raw.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)
baseline_summary.to_csv(
    artifact_path('70_30_groundtruth_clustering_baseline.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)

print('PENDIKOM questionnaire cohort — DBI/SI vs baselines')
display(
    baseline_summary[
        ['Method','Runs','Initial cost','Final cost','DBI','SI','Iterations']
    ].round(4)
)



# ============================================================
# Table 11 — Dimensional performance against independent ILS
# ============================================================

def add_dimension_metrics(rows, method, dimension, truth, pred):
    truth = np.asarray(truth, dtype=object)
    pred = np.asarray(pred, dtype=object)
    labels = sorted(set(truth.tolist()))

    acc = accuracy_score(truth, pred)
    ba = balanced_accuracy_score(truth, pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        truth, pred, labels=labels, average='macro', zero_division=0
    )

    rows.extend([
        {'Method':method,'Measure':'Acc','Dimension':dimension,'Value':float(acc)},
        {'Method':method,'Measure':'BA','Dimension':dimension,'Value':float(ba)},
        {'Method':method,'Measure':'F1','Dimension':dimension,'Value':float(f1)},
    ])


metric_rows = []
care_map_cols = [
    'carekm_active_reflective',
    'carekm_sensing_intuitive',
    'carekm_visual_verbal',
    'carekm_sequential_global',
]

for dim, gt_col, care_col in zip(DIM_NAMES, ILS_CODE_COLS, care_map_cols):
    truth = test_eval[gt_col].map(ils_code_to_full).to_numpy(dtype=object)
    pred = test_eval[care_col].map(pole_abbr_to_full).to_numpy(dtype=object)
    add_dimension_metrics(metric_rows, 'CARE-KM', dim, truth, pred)

train_label = pd.DataFrame({
    'userid': gt_train['userid'].astype(int),
    'cluster_id': ext_train_cluster,
})
train_label['ls_class'] = [
    mp.loc[int(k), 'ls_class'] for k in ext_train_cluster
]
resolved = train_label['ls_class'].ne('AMBIGUOUS').to_numpy()

if resolved.sum() == 0:
    raise RuntimeError('No resolved CARE-KM semantic labels in training data.')

X_train_cls = gt_train.loc[resolved, features].astype(int).reset_index(drop=True)
y_train_cls = train_label.loc[resolved, 'ls_class'].astype(str).reset_index(drop=True)
X_test_cls = gt_test[features].astype(int).reset_index(drop=True)

pred_table = test_eval[
    ['userid'] + ILS_CODE_COLS + ['ils_ls_class']
].copy().reset_index(drop=True)

for model_name in ['NB','DT','SVM']:
    model = build_external_classifiers()[model_name]

    if model_name == 'NB':
        model.fit(X_train_cls.to_numpy(), y_train_cls)
        pred_ls = model.predict(X_test_cls.to_numpy())
    else:
        model.fit(X_train_cls, y_train_cls)
        pred_ls = model.predict(X_test_cls)

    pred_table[f'{model_name}_predicted_ls_class'] = pred_ls

    for di, (dim, gt_col) in enumerate(zip(DIM_NAMES, ILS_CODE_COLS)):
        truth = pred_table[gt_col].map(ils_code_to_full).to_numpy(dtype=object)
        pred_dim = np.asarray(
            [LS_TO_POLES[str(ls)][di] for ls in pred_ls],
            dtype=object,
        )
        pred_table[f'{model_name}_{DIM_KEYS[di]}'] = pred_dim
        add_dimension_metrics(metric_rows, model_name, dim, truth, pred_dim)
        save_confusion(
            truth, pred_dim,
            artifact_path(f'70_30_groundtruth_{model_name}_{DIM_KEYS[di]}_confusion_matrix.csv','22_Holdout_LMS_Pendikom'),
        )

metrics_long = pd.DataFrame(metric_rows)
metrics_long.to_csv(
    artifact_path('70_30_groundtruth_classifier_dimension_metrics.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)
pred_table.to_csv(
    artifact_path('70_30_groundtruth_classifier_predictions.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)

table11_rows = []
for method in ['NB','DT','SVM','CARE-KM']:
    for measure in ['Acc','BA','F1']:
        subset = (
            metrics_long[
                metrics_long['Method'].eq(method) &
                metrics_long['Measure'].eq(measure)
            ]
            .set_index('Dimension')['Value']
        )
        row = {'Method and measure': f'{method} {measure}'}
        for dim in DIM_NAMES:
            row[dim] = float(subset[dim])
        row['Mean'] = float(subset.mean())
        table11_rows.append(row)

table11_numeric = pd.DataFrame(table11_rows)
table11_numeric.to_csv(
    artifact_path('70_30_table11_dimensional_performance_vs_ils_numeric.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)

table11_paper = table11_numeric.copy()
for c in DIM_NAMES + ['Mean']:
    table11_paper[c] = table11_paper[c].map(lambda x: f'{100*x:.2f}%')
table11_paper.to_csv(
    artifact_path('70_30_table11_dimensional_performance_vs_ils.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)

classifier_accuracy_rows = []
for method in ['NB','DT','SVM']:
    subset = (
        metrics_long[
            metrics_long['Method'].eq(method) &
            metrics_long['Measure'].eq('Acc')
        ]
        .set_index('Dimension')['Value']
    )
    classifier_accuracy_rows.append({
        'Classifier': method,
        **{dim: float(subset[dim]) for dim in DIM_NAMES},
        'Mean Dimensional Accuracy': float(subset.mean()),
    })

classifier_table = pd.DataFrame(classifier_accuracy_rows)
classifier_table.to_csv(
    artifact_path('70_30_table_classifier_dimensional_accuracy_vs_ils_numeric.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)
classifier_paper = classifier_table.copy()
for c in DIM_NAMES + ['Mean Dimensional Accuracy']:
    classifier_paper[c] = classifier_paper[c].map(lambda x: f'{100*x:.2f}%')
classifier_paper.to_csv(
    artifact_path('70_30_table_classifier_dimensional_accuracy_vs_ils.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)

care_acc = (
    metrics_long[
        metrics_long['Method'].eq('CARE-KM') &
        metrics_long['Measure'].eq('Acc')
    ]
    .set_index('Dimension')['Value']
)
external_rows = [
    {'FSLSM Dimension': dim, 'Agreement': float(care_acc[dim])}
    for dim in DIM_NAMES
]
external_rows.append({
    'FSLSM Dimension':'Mean dimensional agreement',
    'Agreement':float(care_acc.mean()),
})
external_numeric = pd.DataFrame(external_rows)
external_numeric.to_csv(
    artifact_path('70_30_table_external_validation_carekm_vs_ils_numeric.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)
external_paper = external_numeric.copy()
external_paper['Agreement'] = external_paper['Agreement'].map(
    lambda x: f'{100*x:.2f}%'
)
external_paper.to_csv(
    artifact_path('70_30_table_external_validation_carekm_vs_ils.csv','22_Holdout_LMS_Pendikom'),
    index=False,
)

print('Resolved train labels:', int(resolved.sum()), '/', len(resolved))
print('Table 11. Dimensional performance against independent ILS labels')
display(table11_paper)



care_row = baseline_summary.loc[
    baseline_summary['Method'].eq('CARE-KM')
].iloc[0]

summary = {
    'protocol': (
        'PENDIKOM WITH_ILS cohort; frozen 322/139 split; train-only stay references; '
        'CARE-KM and semantic mapping fitted on train; 139 holdout evaluated against ILS'
    ),
    'source': str(PENDIKOM_CATEGORICAL_INPUT),
    'repository': 'https://github.com/Dewi2501/Learning-Style-Based',
    'selection': 'has_ils == 1',
    'source_rows': int(len(pendikom_ils)),
    'historical_compiled_source_rows': PENDIKOM_SOURCE_ROWS,
    'independent_ils_reference': str(_ROOT_REVISED/'LIB/PROTOKOL/pendikom_ils_labels.csv'),
    'ils_rows': int(len(pendikom_ils)),
    'train_rows': int(len(ils_train)),
    'test_rows': int(len(ils_test)),
    'split_file': str(GROUNDTRUTH_SPLIT),
    'split_seed_reference': SPLIT_SEED,
    'core_features': int(len(PENDIKOM_FEATURES)),
    'used_features': int(len(features)),
    'constant_features': constant_features,
    'resolved_train_learners': int(resolved.sum()),
    'unresolved_train_learners': int((~resolved).sum()),
    'care_train_dbi': float(care_row['DBI']),
    'care_train_si': float(care_row['SI']),
    'care_config': CARE_CONFIG,
    'parent_initializer': 'Cao',
    'refinement_budget_rule': 'ceil(log2(candidate_budget))',
    'derived_refinement_budget': int(DERIVED_REFINEMENT_BUDGET),
    'exchange_depth_is_independent_hyperparameter': False,
    'care_implementation_version': str(ck.CARE_IMPLEMENTATION_VERSION),
    'groundtruth_usage': (
        'ILS labels do not form CARE-KM clusters or semantic mapping; '
        'they are used as the independent questionnaire reference.'
    ),
}
(artifact_path('70_30_locked_protocol_summary.json','22_Holdout_LMS_Pendikom')).write_text(
    json.dumps(summary, indent=2), encoding='utf-8'
)
print(json.dumps(summary, indent=2))


In [ ]:
# Akurasi, balanced accuracy, dan macro-F1 dari prediksi holdout LMS.
from experiment_lib import main_lms_classifier_metrics
main_classifier_metrics=main_lms_classifier_metrics(_ROOT_REVISED)
main_classifier_metrics.to_csv(artifact_path('70_30_main_lms_classifier_metrics.csv'),index=False)
display(main_classifier_metrics)
